# Deployment

> The power control in layouts near the training layouts and far from them, with its margin chosen once (offline),
> per layout from a pilot, or adapted online from the acknowledgements.

In [ ]:
#| default_exp comm.deployment

In [ ]:
#| hide
from nbdev.showdoc import *

`comm.evaluation` judges the power control on test layouts drawn like the training ones. At deployment, a layout can
be close to one the predictor was trained on, or unlike any of them. This module groups the deployment layouts by
that **shift**, the normalized edit distance to the nearest training layout (`comm.channel_data.layout_distance`),
and reports every metric per group. It adds two ways to choose the margin `κ` in the deployment layout itself:

* **per-layout calibration**: the first `pilot` episodes in a layout measure the channel (the receivers report the
  SNR they get), and `κ` is chosen on them: the least margin whose reliability on the pilot reaches the target,
  by a lower confidence bound (a pilot is short, and its agents visit only part of the layout);
* **κ and τ over layouts** (`comm.power_control.calibrate_layouts`): chosen offline, on the calibration layouts, for
  a share of new layouts to reach the target rather than their transmissions pooled;
* **online adaptation**: from the start, `κ` follows the acknowledgements, as outer-loop link adaptation does: up
  `δ` after a failure, down `δ (1 − p) / p` after a success, which settles where a share `p` of the transmissions
  succeeds. A failure at full power says nothing about the margin (more of it could not have been sent), so it is
  not counted. Withholding is decided with the offline `κ` and `τ`, so a growing `κ` does not withhold more (and
  then see fewer failures). `δ` and `p` are chosen on the calibration layouts, as `κ` and `τ` are.

Every method is judged on the same transmissions: the deployment layouts' episodes after the pilot.

In [ ]:
#| export
from __future__ import annotations

import json
import math
from pathlib import Path

import numpy as np

from c3jepa_wm.comm.channel_data import interior_walls, layout_edits, make_env, nearest_layout, reset_layout
from c3jepa_wm.comm.evaluation import LEARNED, baselines, layout_intervals, prediction_errors, transmissions
from c3jepa_wm.comm.power_control import MARGINS_DB, TAUS_DB, calibrate, calibrate_layouts, predict_gain, predict_on

## The deployment layouts

`deployment_layouts` builds `per_group` layouts for each group, a band of distance `[lo, hi]` to the nearest
training layout:

* the groups in `edited` (by default *in-distribution* and *slightly OOD*): a training layout with `k` of its cells
  flipped (`layout_edits`), `k` drawn for a distance in the band;
* the others (*OOD*): new layouts of the generator (reset seeds from `pool`) whose distance falls in the band.

Every layout's distance is measured, not assumed: it is kept only if it falls in its group's band. The edited
layouts' bases are chosen with the wall counts of the new layouts, so the groups differ in their distance to the
training layouts, not in how crowded they are (the number of walls also decides how hard a layout is).

In [ ]:
#| export
GROUPS = {'in-distribution': (0.0, 0.15), 'slightly OOD': (0.2, 0.4), 'OOD': (0.55, 1.0)}
EDITED = ('in-distribution', 'slightly OOD')

def _edits_for(distance: float, walls: int) -> int:
    "Edits (half walls removed, half added) that put a layout of `walls` walls at about `distance` from itself."
    return max(1, int(round(2 * distance * walls / (2 - distance))))

def deployment_layouts(
    env_cfg, # The environment's config (`make_env`)
    train, # The training layouts' reset seeds
    pool, # Reset seeds to draw the new layouts from (the training seeds are skipped)
    groups: dict = GROUPS, # Group -> its band [lo, hi] of normalized distance to the nearest training layout
    edited=EDITED, # The groups made by editing training layouts (the others: new layouts)
    per_group: int = 30, # Layouts per group
    seed: int = 0, # Of the edits
    first_id: int = 1_000_000, # The edited layouts' ids: first_id, first_id + 1, ...
    tries: int = 50, # Edits tried per layout before giving up
) -> dict:
    """`layouts`: id -> its `group`, `base` seed and flipped `cells` (edited; None and [] for a new layout), `distance`
    (normalized) and `edits` to the `nearest` training layout, and its `walls`; `groups`: group -> its ids."""
    env = make_env(env_cfg)
    train = [int(s) for s in train]
    reference = np.stack([interior_walls(reset_layout(env, s)) for s in train])
    train_walls = reference.reshape(len(train), -1).sum(1)
    rng = np.random.default_rng(seed)
    layouts, members = {}, {g: [] for g in groups}
    fresh = (int(s) for s in pool if int(s) not in set(train))
    for g in (g for g in groups if g not in edited):
        lo, hi = groups[g]
        while len(members[g]) < per_group:
            s = next(fresh, None)
            if s is None:
                raise ValueError(f"the pool ran out with {len(members[g])} of {per_group} layouts in {g!r} {groups[g]}")
            w = interior_walls(reset_layout(env, s))
            i, e, d = nearest_layout(w, reference)
            if lo <= d <= hi:
                layouts[s] = dict(group=g, base=None, cells=[], distance=d, edits=e, nearest=train[i], walls=int(w.sum()))
                members[g].append(s)
    match = [layouts[s]['walls'] for g in members if g not in edited for s in members[g]] or train_walls.tolist()
    next_id = int(first_id)
    for g in (g for g in groups if g in edited):
        lo, hi = groups[g]
        for n in range(per_group):
            near = np.argsort(np.abs(train_walls - match[n % len(match)]) + rng.random(len(train)))[:5]
            for _ in range(tries):
                b = int(rng.choice(near))
                reset_layout(env, train[b])
                try:
                    cells = layout_edits(env, _edits_for(rng.uniform(lo, hi), int(train_walls[b])), rng)
                except RuntimeError:
                    continue
                w = interior_walls(reset_layout(env, next_id, {next_id: dict(base=train[b], cells=cells)}))
                i, e, d = nearest_layout(w, reference)
                if lo <= d <= hi:
                    break
            else:
                raise RuntimeError(f"no edits of the training layouts land in {g!r} {groups[g]} in {tries} tries")
            layouts[next_id] = dict(group=g, base=train[b], cells=cells, distance=d, edits=e, nearest=train[i],
                                    walls=int(w.sum()))
            members[g].append(next_id)
            next_id += 1
    return dict(layouts=layouts, groups=members, train=train, bands={g: list(b) for g, b in groups.items()})

def layout_edits_of(deployment: dict) -> dict:
    "The edited layouts of a deployment (`reset_layout`'s `edits`): id -> its base seed and flipped cells."
    return {k: dict(base=v['base'], cells=v['cells']) for k, v in deployment['layouts'].items() if v['base'] is not None}

def save_deployment(path, deployment: dict):
    "Write a deployment to JSON (atomically: a SLURM array's shards may all write the same one)."
    path = Path(path)
    tmp = path.with_suffix(f'.{np.random.default_rng().integers(1 << 30)}.tmp')
    tmp.write_text(json.dumps(deployment, indent=1))
    tmp.replace(path)

def load_deployment(path) -> dict:
    "A deployment from `save_deployment`'s JSON (with integer ids)."
    d = json.loads(Path(path).read_text())
    d['layouts'] = {int(k): v for k, v in d['layouts'].items()}
    return d

def deployment_summary(deployment: dict) -> str:
    "A text table: per group, its band, layouts, and the distance, edits and walls (mean, min-max)."
    lines = [f"{'group':18s}{'band':>14s}{'layouts':>9s}{'distance':>20s}{'edits':>16s}{'walls':>16s}"]
    for g, ids in deployment['groups'].items():
        v = {k: np.array([deployment['layouts'][i][k] for i in ids], float) for k in ('distance', 'edits', 'walls')}
        lo, hi = deployment['bands'][g]
        lines.append(f"{g:18s}{f'[{lo:g}, {hi:g}]':>14s}{len(ids):9d}" + ''.join(
            f"{v[k].mean():9.2f} ({v[k].min():.2f}-{v[k].max():.2f})" if k == 'distance' else
            f"{v[k].mean():7.1f} ({v[k].min():.0f}-{v[k].max():.0f})" for k in ('distance', 'edits', 'walls')))
    lines.append('distance: the edits to the nearest training layout over the walls in either (0: the same walls, 1: none in common)')
    return '\n'.join(lines)

In [ ]:
show_doc(deployment_layouts)

## Choosing the margin in the deployment layout

`per_layout_margins` chooses `κ` per layout on its pilot transmissions (`margin_for`), with the transmissions withheld
as at deployment (by the offline `κ` and `τ`); a layout whose pilot has fewer than `min_servable` servable
transmissions keeps the offline `κ`. `online_margins` replays each layout's
transmissions in the order they are sent (episode, step, link) and returns the margin each is sent with; it runs
several (`δ`, `p`) at once, so `choose_online` tries them all on the calibration layouts in one pass.

In [ ]:
#| export
def sequences(data: dict, layouts=None) -> tuple[list, np.ndarray]:
    "(layouts, index): each layout's samples in the order they are sent (episode, step, link), a row each, -1 padded."
    order = np.lexsort((data['link'][:, 0], data['link'][:, 2], data['episode'], data['layout']))
    layout = data['layout'][order]
    layouts = list(np.unique(layout)) if layouts is None else list(layouts)
    rows = [order[layout == k] for k in layouts]
    index = np.full((len(rows), max(map(len, rows), default=0)), -1)
    for r, idx in enumerate(rows):
        index[r, :len(idx)] = idx
    return [int(k) for k in layouts], index

def margin_for(estimate, gain, sent, g_min: float, target: float, margins_db=MARGINS_DB,
               confidence: float | None = None) -> tuple[float, bool]:
    """(κ, met): the least-power margin with which the `sent` transmissions reach `target` of the servable ones
    (the withholding given), else the least margin with the most reliability. With `confidence`, the reliability's
    lower confidence bound (Clopper-Pearson) must reach `target`, not the reliability itself: few transmissions
    then need a larger margin."""
    servable = gain >= g_min
    m = np.asarray(margins_db, float)[:, None]
    power = np.where(sent, np.minimum(10 ** (m / 10) / np.maximum(estimate, 1e-30), 1 / g_min), 0.0)
    reached = (power >= 1 / gain * (1 - 1e-9))[:, servable]
    reliability = reached.mean(1)
    if confidence is None:
        met = reliability >= target
    else:
        from scipy.stats import beta
        x, n = reached.sum(1), servable.sum()
        met = np.where(x > 0, beta.ppf(1 - confidence, np.maximum(x, 1), n - x + 1), 0.0) >= target
    if met.any():
        return float(m[met, 0][np.argmin(power[met].mean(1))]), True
    return float(m[np.argmax(reliability), 0]), False

def per_layout_margins(
    estimate, gain, layout, # ĝ, |h|² and layout of each transmission
    pilot: np.ndarray, # The transmissions of the pilot (boolean)
    g_min: float, target: float, tau_db: float, # The weakest servable gain, the target, the offline τ
    fallback_db: float, # The offline κ: the withholding, and the margin of a layout with too short a pilot
    min_servable: int = 50, # Servable pilot transmissions needed
    margins_db=MARGINS_DB, # Margins tried
    confidence: float | None = 0.9, # Of the reliability's lower bound on the pilot (`margin_for`); None: the reliability itself
) -> dict:
    """Layout -> its κ (dB): the least-power margin reaching `target` on its pilot transmissions (`margin_for`), those
    withheld as at deployment (by the offline κ and τ); if none does, the most reliable one but at least the offline κ."""
    _, sent = transmissions(estimate, g_min, fallback_db, tau_db)
    out = {}
    for k in np.unique(layout):
        on = (layout == k) & pilot
        if (gain[on] >= g_min).sum() < min_servable:
            out[int(k)] = float(fallback_db)
            continue
        kappa, met = margin_for(estimate[on], gain[on], sent[on], g_min, target, margins_db, confidence)
        out[int(k)] = kappa if met else max(kappa, float(fallback_db))
    return out

def online_margins(
    estimate, gain, # ĝ and |h|² of every transmission
    index: np.ndarray, # `sequences`' index: each layout's transmissions in order
    g_min: float, # The weakest servable gain
    margin_db: float, tau_db: float, # The offline κ (the start, and the withholding) and τ
    steps_db, success, # δ (dB) and the success share p aimed at, one per run (broadcast together)
    bounds_db=(-10.0, 30.0), # κ kept within the offline κ + these
) -> np.ndarray:
    """The margin (dB) each transmission is sent with, per run (runs, transmissions; NaN if not in `index`): from the
    offline κ, up δ after a failure (not at full power), down δ (1 − p) / p after a success, per layout."""
    steps, success = np.broadcast_arrays(np.atleast_1d(np.asarray(steps_db, float)), np.atleast_1d(np.asarray(success, float)))
    down = steps * (1 - success) / success
    runs, (L, T) = len(steps), index.shape
    kappa = np.full((runs, L), float(margin_db))
    lo, hi = margin_db + bounds_db[0], margin_db + bounds_db[1]
    out = np.full((runs, len(gain)), np.nan)
    cap = 1 / g_min
    for t in range(T):
        idx = index[:, t]
        on = idx >= 0
        if not on.any():
            break
        i = idx[on]
        est, g = np.maximum(estimate[i], 1e-30), gain[i]
        out[:, i] = kappa[:, on]
        sent = 10 ** (margin_db / 10) / est <= (10 ** (tau_db / 10) * cap if math.isfinite(tau_db) else np.inf)
        rho = 10 ** (kappa[:, on] / 10) / est
        power = np.minimum(rho, cap)
        ok = power * g >= 1 - 1e-9
        counted = sent & (ok | (rho < cap))                      # a failure at full power says nothing of κ
        kappa[:, on] = np.clip(kappa[:, on] + np.where(counted, np.where(ok, -down[:, None], steps[:, None]), 0.0), lo, hi)
    return out

def _sent_with(estimate, g_min: float, margin_db, tau_db: float, schedule_db: float) -> tuple:
    "(power in units of Γσ², sent): sent at `margin_db` (per transmission) when the offline `schedule_db` and τ send it."
    _, sent = transmissions(estimate, g_min, schedule_db, tau_db)
    power = np.where(sent, np.minimum(10 ** (np.asarray(margin_db) / 10) / np.maximum(estimate, 1e-30), 1 / g_min), 0.0)
    return power, sent

In [ ]:
#| export
def judge(power, sent, gain, layout, g_min: float, target: float | None = None, min_servable: int = 20, episode=None) -> dict:
    """The share of the servable transmissions reaching Γ (`reliability`), the power (× perfect CSI's), the share
    withheld, and the reliability per layout (over the layouts with at least `min_servable`): mean, std, 10th
    percentile, min, and with `target` the share of the layouts below it; with each transmission's `episode`, also
    the share significantly below it and the std between layouts without the sampling noise (`layout_intervals`)."""
    servable, need = gain >= g_min, 1 / np.maximum(gain, 1e-300)
    reached = sent & (power >= need * (1 - 1e-9))
    out = dict(reliability=float(reached[servable].mean()) if servable.any() else math.nan,
               power=float(power.mean() / np.where(servable, need, 0).mean()) if servable.any() else math.nan,
               withheld=float(1 - sent.mean()), servable=float(servable.mean()), transmissions=int(len(gain)))
    per = np.array([reached[(layout == k) & servable].mean() for k in np.unique(layout)
                    if ((layout == k) & servable).sum() >= min_servable])
    out['layouts'] = int(len(per))
    if len(per):
        out.update(layout_mean=float(per.mean()), layout_std=float(per.std()), layout_p10=float(np.percentile(per, 10)),
                   layout_min=float(per.min()))
        if target is not None:
            out['layout_below'] = float((per < target).mean())
        if episode is not None:
            keep = [int(k) for k in np.unique(layout) if ((layout == k) & servable).sum() >= min_servable]
            iv = layout_intervals(reached, servable, layout, np.asarray(episode))
            se = np.array([iv[k][2] for k in keep])
            out['layout_between_std'] = float(math.sqrt(max(per.var() - (se ** 2).mean(), 0.0)))
            if target is not None:
                out['layout_significantly_below'] = float(np.mean([iv[k][1] < target for k in keep]))
    return out

def choose_online(
    estimate, gain, data: dict, # ĝ and |h|² of the calibration transmissions, and their samples (`episode`, `link`, `layout`)
    g_min: float, target: float, margin_db: float, tau_db: float, # The weakest servable gain, the target, the offline κ and τ
    pilot: int = 2, # Episodes before the ones judged (adapted through, as at deployment)
    steps_db=(0.25, 0.5, 1.0, 2.0), # δ tried
    failure_scales=(1.0, 0.75, 0.5, 0.25), # p tried: 1 - (1 - target) × each
    bounds_db=(-10.0, 30.0), # As `online_margins`'
) -> dict:
    "The δ and p with the least power reaching `target` on the calibration layouts (else the most reliable)."
    steps, scales = np.meshgrid(np.asarray(steps_db, float), np.asarray(failure_scales, float), indexing='ij')
    steps, success = steps.ravel(), 1 - (1 - target) * scales.ravel()
    _, index = sequences(data)
    margins = online_margins(estimate, gain, index, g_min, margin_db, tau_db, steps, success, bounds_db)
    judged = data['episode'] >= pilot
    best = None
    for r in range(len(steps)):
        power, sent = _sent_with(estimate[judged], g_min, margins[r, judged], tau_db, margin_db)
        o = judge(power, sent, gain[judged], data['layout'][judged], g_min)
        met = o['reliability'] >= target
        key = (met, -o['power'] if met else o['reliability'])
        if best is None or key > best[0]:
            best = (key, dict(step_db=float(steps[r]), success=float(success[r]), met=bool(met),
                              reliability=o['reliability'], power=o['power']))
    return best[1]

In [ ]:
show_doc(online_margins)

In [ ]:
# online_margins settles where a share p of the counted transmissions succeeds, from any offset of the predictor
rng = np.random.default_rng(0)
n, g_min = 6000, 1e-3
gain = 10 ** (rng.uniform(-28, -10, n) / 10)                                   # all servable (≥ g_min): no full-power failures
estimate = gain * 10 ** ((6 + 4 * rng.standard_normal(n)) / 10)                # 6 dB too optimistic, 4 dB spread
data = dict(link=np.c_[np.zeros(n, int), np.zeros(n, int), np.arange(n)], episode=np.zeros(n, int), layout=np.zeros(n, int))
_, index = sequences(data)
margins = online_margins(estimate, gain, index, g_min, 0.0, math.inf, [0.5, 0.5], [0.9, 0.99])
for p, m in zip((0.9, 0.99), margins):
    power, sent = _sent_with(estimate, g_min, m, math.inf, 0.0)
    late = slice(n // 2, None)
    assert abs((power * gain >= 1 - 1e-9)[late].mean() - p) < 0.015, p
assert margins[1, -1] > margins[0, -1] > 6                                     # above the predictor's offset, more for 99%
# the order of a layout's transmissions: episode, then step, then link
data = dict(link=np.array([[1, 0, 5], [0, 1, 5], [0, 1, 4], [0, 1, 9]]), episode=np.array([1, 1, 1, 0]), layout=np.array([3, 3, 3, 3]))
assert sequences(data)[1].tolist() == [[3, 2, 1, 0]]
# margin_for: the least-power margin meeting the target on these transmissions, the withholding given
sent = rng.random(n) < 0.9
kappa, met = margin_for(estimate, gain, sent, g_min, 0.8)
power = lambda k: _sent_with(estimate, g_min, np.full(n, k), math.inf, 0.0)[0] * sent
assert met and ((power(kappa) * gain >= 1 - 1e-9)[sent].mean() * sent.mean() >= 0.8 - 1e-9)
assert ((power(kappa - 0.25) * gain >= 1 - 1e-9).mean() < 0.8)
kappa, met = margin_for(estimate, gain, sent, g_min, 0.95)                     # 10% withheld: 95% out of reach
assert not met and (power(kappa) * gain >= 1 - 1e-9)[sent].all()                # the least margin reaching all those sent
few = slice(0, 60)                                                             # by the lower bound: a larger margin, more so on few
k_all, k_conf, k_few = (margin_for(estimate[i], gain[i], sent[i], g_min, 0.8, confidence=c)[0]
                        for i, c in ((slice(None), None), (slice(None), 0.9), (few, 0.9)))
assert k_all <= k_conf <= k_few and k_all < k_few

## Per group

`evaluate_deployment` chooses each method's `κ` and `τ` per target on the calibration layouts, then judges every
method on the deployment layouts after the pilot, per group and over all of them (`all`). The methods: every
predictor of `comm.evaluation` with its offline `κ` and `τ`, and for the predictors in `adapt`, the same with the
margin chosen per layout (`+ per-layout κ`) and adapted online (`+ online κ`).

In [ ]:
#| export
PER_LAYOUT, ONLINE, OVER_LAYOUTS = ' + per-layout κ', ' + online κ', ' + κ, τ over layouts'

def evaluate_deployment(
    saved: dict, # `load_power_control`'s result
    data: dict, # {'train', 'calibration', 'deployment'}: `build_dataset`'s (with `episode`)
    deployment: dict, # `deployment_layouts`' result (the groups)
    fitted: dict | None = None, # `comm.evaluation.fit_baselines`' result (None: fit here)
    targets=(0.9, 0.95, 0.99), # Reliability targets
    taus_db=TAUS_DB, # Thresholds tried in the calibration
    slack: float = 0.0, # Calibrate for target + slack
    pilot: int = 2, # Episodes of each deployment layout before the ones judged
    confidence: float | None = 0.9, # Per-layout: of the reliability's lower bound on the pilot (None: the reliability)
    adapt=('geometry + history mean', LEARNED), # Predictors also with per-layout and online κ
    min_servable: int = 50, # Servable pilot transmissions per-layout calibration needs
    steps_db=(0.25, 0.5, 1.0, 2.0), # Online: δ tried on the calibration layouts
    failure_scales=(1.0, 0.75, 0.5, 0.25), # Online: p tried, 1 - (1 - target) × each
    bounds_db=(-10.0, 30.0), # Online: κ within the offline κ + these
    quantile: float = 0.05, max_iter: int = 400, # Of the baselines, if fit here
    layout_coverage: float | None = None, # Also κ and τ chosen over the calibration layouts (`calibrate_layouts`), for this share
    layout_confidence: float | None = None, # ... with this probability over the calibration layouts (None: in expectation)
) -> dict:
    """`groups`: per group its layouts, distance, edits and walls; `methods`: per method its predictor's errors per
    group, and per target its parameters and `judge`'s metrics per group (and `all`)."""
    budget, g_min = saved['budget'], saved['budget'].g_min
    cal, dep = data['calibration'], data['deployment']
    predictors = {LEARNED: lambda d: predict_on(saved['model'], d),
                  **baselines(data['train'], saved['history'], quantile, max_iter, fitted)}
    meta = deployment['layouts']
    groups = {g: [k for k in ids if k in set(dep['layout'].tolist())] for g, ids in deployment['groups'].items()}
    judged = dep['episode'] >= pilot
    member = {'all': judged, **{g: judged & np.isin(dep['layout'], ids) for g, ids in groups.items()}}
    out = dict(pilot=pilot, transmissions=int(judged.sum()),
               groups={g: dict(layouts=len(ids), **{k: float(np.mean([meta[i][k] for i in ids])) for k in ('distance', 'edits', 'walls')})
                       for g, ids in groups.items()},
               methods={})

    def metrics(power, sent, target):
        return {g: judge(power[m], sent[m], dep['gain'][m], dep['layout'][m], g_min, target, episode=dep['episode'][m])
                for g, m in member.items()}

    for name, predict in predictors.items():
        c_est, d_est = predict(cal), predict(dep)
        errors = {g: prediction_errors(d_est[m], dep['gain'][m]) for g, m in member.items()}
        kinds = {name: {}}
        if name in adapt and name != 'perfect CSI':
            kinds[name + PER_LAYOUT], kinds[name + ONLINE] = {}, {}
            if layout_coverage is not None:
                kinds[name + OVER_LAYOUTS] = {}
        for target in targets:
            if name == 'perfect CSI':
                offline = dict(margin_db=0.0, tau_db=0.0, met=True)
            else:
                offline = calibrate(c_est, cal['gain'], g_min, target + slack, taus_db=taus_db)
            kappa, tau = offline['margin_db'], offline['tau_db']
            power, sent = transmissions(d_est, g_min, kappa, tau)
            kinds[name][target] = dict(margin_db=kappa, tau_db=tau, met_on_calibration=offline['met'],
                                       groups=metrics(power, sent, target))
            if name + PER_LAYOUT in kinds:
                margins = per_layout_margins(d_est, dep['gain'], dep['layout'], ~judged, g_min, target, tau, kappa,
                                             min_servable, confidence=confidence)
                power, sent = _sent_with(d_est, g_min, np.array([margins[int(k)] for k in dep['layout']]), tau, kappa)
                m = np.array(list(margins.values()))
                kinds[name + PER_LAYOUT][target] = dict(margin_db=kappa, tau_db=tau, layout_margin_mean_db=float(m.mean()),
                                                        layout_margin_std_db=float(m.std()), per_layout_db=margins,
                                                        groups=metrics(power, sent, target))
                chosen = choose_online(c_est, cal['gain'], cal, g_min, target + slack, kappa, tau, pilot, steps_db,
                                       failure_scales, bounds_db)
                _, index = sequences(dep)
                margin = online_margins(d_est, dep['gain'], index, g_min, kappa, tau, chosen['step_db'], chosen['success'],
                                        bounds_db)[0]
                power, sent = _sent_with(d_est, g_min, margin, tau, kappa)
                kinds[name + ONLINE][target] = dict(margin_db=kappa, tau_db=tau, step_db=chosen['step_db'],
                                                    success=chosen['success'], met_on_calibration=chosen['met'],
                                                    final_margin_mean_db=float(np.nanmean([margin[i[i >= 0][-1]] for i in index])),
                                                    groups=metrics(power, sent, target))
            if name + OVER_LAYOUTS in kinds:
                ch = calibrate_layouts(c_est, cal['gain'], cal['layout'], g_min, target, layout_coverage, layout_confidence,
                                       taus_db=taus_db)
                power, sent = transmissions(d_est, g_min, ch['margin_db'], ch['tau_db'])
                kinds[name + OVER_LAYOUTS][target] = dict(margin_db=ch['margin_db'], tau_db=ch['tau_db'], met_on_calibration=ch['met'],
                                                          calibration_below=ch['below'], allowed=ch['allowed'],
                                                          groups=metrics(power, sent, target))
        for method, per_target in kinds.items():
            out['methods'][method] = dict(predictor=name, prediction=errors, targets=per_target)
    return out

In [ ]:
#| export
def deployment_table(results: dict, target: float, keys=('reliability', 'power', 'layout_p10', 'layout_below')) -> str:
    "A text table at one target: per method, per group (and all), the reliability, power, 10th-percentile layout and the layouts below target."
    groups = ['all', *results['groups']]
    names = list(results['methods'])
    w = max(map(len, names)) + 2
    head = f"{'':{w}s}" + ''.join(f"{g:>44s} " for g in groups) + '\n' + f"{'':{w}s}" + ''.join(
        f"{'reached':>21s}{'power':>8s}{'p10':>8s}{'below':>7s} " for _ in groups)
    lines = [f"target {100 * target:g}%, judged after a pilot of {results['pilot']} episodes ({results['transmissions']} transmissions)",
             *(f"  {g}: {v['layouts']} layouts, distance {v['distance']:.2f}, {v['edits']:.1f} edits, {v['walls']:.0f} walls"
               for g, v in results['groups'].items()), head]
    for n in names:
        row = f"{n:{w}s}"
        for g in groups:
            o = results['methods'][n]['targets'][target]['groups'][g]
            row += (f"{100 * o['reliability']:20.1f}% {o['power']:6.2f}× {100 * o.get('layout_p10', math.nan):6.1f}% "
                    f"{100 * o.get('layout_below', math.nan):5.0f}% ")
        lines.append(row)
    lines.append('p10: the 10th-percentile layout\'s reliability; below: the share of layouts under the target')
    return '\n'.join(lines)

def plot_deployment(results: dict, target: float, path=None, methods=None):
    """Per group: each method's reliability (pooled, and its 10th-percentile layout) and power, at one target: a colour
    per predictor, dotted with the offline κ, solid per layout, dashed online."""
    import matplotlib.pyplot as plt
    groups = list(results['groups'])
    methods = methods or [n for n in results['methods'] if n != 'perfect CSI']
    fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4.2))
    x = np.arange(len(groups))
    predictors = list(dict.fromkeys(results['methods'][n]['predictor'] for n in methods))
    colors = {p: plt.cm.tab10(i % 10) for i, p in enumerate(predictors)}       # a colour per predictor, a style per way of choosing κ
    for k, n in enumerate(methods):
        o = [results['methods'][n]['targets'][target]['groups'][g] for g in groups]
        kind = 'per-layout' if n.endswith(PER_LAYOUT) else 'online' if n.endswith(ONLINE) else 'offline'
        style = dict(marker={'offline': '.', 'per-layout': 'o', 'online': 's'}[kind], ms=6, lw=1.5, label=n,
                     ls={'offline': ':', 'per-layout': '-', 'online': '--'}[kind], color=colors[results['methods'][n]['predictor']])
        dx = 0.03 * (k - len(methods) / 2)
        a.plot(x + dx, [100 * v['reliability'] for v in o], **style)
        a.plot(x + dx, [100 * v.get('layout_p10', math.nan) for v in o], marker='v', ms=4, ls='none', color=style['color'], alpha=0.5)
        b.plot(x, [v['power'] for v in o], **style)
    a.axhline(100 * target, color='#d62728', lw=1, ls='--')
    a.set(ylabel='servable transmissions reaching Γ (%)', title=f'Reliability, target {100 * target:g}% (▼: 10th-percentile layout)')
    b.set(ylabel='power (× perfect CSI)', yscale='log', title='Power')
    for ax in (a, b):
        ax.set_xticks(x, groups)
        ax.spines[['top', 'right']].set_visible(False)
    b.legend(frameon=False, fontsize=7, loc='upper left', bbox_to_anchor=(1, 1))
    fig.tight_layout()
    if path is not None:
        fig.savefig(path, dpi=120, bbox_inches='tight')
    return fig

## Example

The 9 × 9 stand-in layouts of `comm.channel_data`'s example (free-space loss and 6 dB per wall, no ray tracing):
six training layouts, three for calibration, and two deployment layouts per group.

In [ ]:
import tempfile

from c3jepa_wm.comm.channel_data import (ChannelTable, PairCache, build_dataset, collect_trajectories, edit_distance,
                                         layout_distance, trace_visited_pairs, wall_geometry)
from c3jepa_wm.comm.evaluation import fit_baselines
from c3jepa_wm.comm.power_control import Budget, fit_predictor

env_cfg = {'_target_': 'stable_marl.envs.multigrid.FindGoalEnv', 'agents': 2, 'size': 9, 'num_obstacles': 1, 'n_clutter': 6,
           'max_steps': 60, 'variations': ['obstacles.number', 'clutter.number']}
tmp = Path(tempfile.mkdtemp())
train, calibration = list(range(6)), list(range(6, 9))
deployment = deployment_layouts(env_cfg, train, range(9, 400), {'in-distribution': (0.0, 0.2), 'slightly OOD': (0.25, 0.45),
                                                               'OOD': (0.55, 1.0)}, per_group=2)
print(deployment_summary(deployment))

In [ ]:
edits = layout_edits_of(deployment)
deployed = [k for ids in deployment['groups'].values() for k in ids]
env = make_env(env_cfg)
reference = np.stack([interior_walls(reset_layout(env, s)) for s in train])
for k, v in deployment['layouts'].items():                                     # every distance as measured, in its band
    w = interior_walls(reset_layout(env, k, edits))
    lo, hi = deployment['bands'][v['group']]
    assert lo <= v['distance'] <= hi and abs(layout_distance(w[None], reference).min() - v['distance']) < 1e-12
    assert (v['base'] is None) == (v['group'] == 'OOD') and (v['base'] is None or v['base'] in train)
assert deployment_layouts(env_cfg, train, range(9, 400), deployment['bands'], per_group=2)['layouts'].keys() == deployment['layouts'].keys()
save_deployment(tmp / 'deployment.json', deployment)
assert load_deployment(tmp / 'deployment.json')['layouts'] == deployment['layouts']

def stand_in(walls):
    def tracer(tx, rxs):
        return [0.125 / (4 * np.pi * 2 * d) * 10 ** (-6 * w / 20) for d, w in (wall_geometry(walls, tx, rx) for rx in rxs)]
    return tracer

collect_trajectories(env_cfg, train + calibration, tmp, episodes=4, max_steps=50)
collect_trajectories(env_cfg, deployed, tmp, episodes=5, max_steps=50, edits=edits)
for k in train + calibration + deployed:
    trace_visited_pairs(env_cfg, tmp, [k], tmp, tracer=stand_in(PairCache(tmp / 'unused.npz', env_cfg, k, edits=edits).walls),
                        edits=edits, progress=False)
sets = {name: build_dataset(tmp, tmp, layouts, fading='rayleigh') for name, layouts in
        (('train', train), ('calibration', calibration), ('deployment', deployed))}
budget = Budget(max_power_dbm=-5.0)
model = fit_predictor(sets['train']['X'], sets['train']['gain'], 0.05, max_iter=50)
saved = dict(model=model, history=4, budget=budget)
results = evaluate_deployment(saved, sets, deployment, fit_baselines(sets['train'], 4, max_iter=50), targets=(0.9, 0.95), pilot=2,
                              min_servable=20, layout_coverage=0.5)
print(deployment_table(results, 0.9))

In [ ]:
plot_deployment(results, 0.9);

In [ ]:
m = results['methods']
assert set(results['groups']) == {'in-distribution', 'slightly OOD', 'OOD'}
assert {LEARNED + PER_LAYOUT, LEARNED + ONLINE, 'geometry + history mean' + ONLINE} <= set(m)
assert 'perfect CSI' + ONLINE not in m and m['perfect CSI']['targets'][0.9]['groups']['all']['reliability'] == 1
for name, r in m.items():
    for t, o in r['targets'].items():
        g = o['groups']
        n = sum(g[k]['transmissions'] for k in results['groups'])
        assert n == g['all']['transmissions'] == results['transmissions']          # the groups split the judged transmissions
        pooled = sum(g[k]['reliability'] * g[k]['servable'] * g[k]['transmissions'] for k in results['groups'])
        assert abs(pooled / (g['all']['servable'] * n) - g['all']['reliability']) < 1e-9
static = m[LEARNED]['targets'][0.9]
per = m[LEARNED + PER_LAYOUT]['targets'][0.9]
assert set(per['per_layout_db']) == set(deployed) and per['tau_db'] == static['tau_db']
over = m[LEARNED + OVER_LAYOUTS]['targets'][0.9]
assert over['allowed'] == 1 and (over['calibration_below'] <= 1 or not over['met_on_calibration'])
g = m[LEARNED]['targets'][0.9]['groups']['all']
assert g['layout_significantly_below'] <= g['layout_below'] and g['layout_between_std'] >= 0
on = m[LEARNED + ONLINE]['targets'][0.9]
assert on['step_db'] in (0.25, 0.5, 1.0, 2.0) and 0.9 <= on['success'] < 1